# Module 4: Full-Stack Local AI Pipeline (Capstone Integration)
**หลักสูตร:** การพัฒนาเวิร์กโฟลว์ปัญญาประดิษฐ์สำหรับสร้างข้อความและรูปภาพบนเครื่องคอมพิวเตอร์ส่วนบุคคล  
**(Mastering Local AI Stack: Jupyter Notebook × LM Studio × ComfyUI)**

---

## 🏗️ สถาปัตยกรรมการเชื่อมโยงระบบ (End-to-End Local AI Architecture)

ในบทเรียน Capstone นี้ เราจะใช้ **Jupyter Notebook** เป็นตัวกลางควบคุม (Orchestrator) เพื่อเชื่อมโยงการทำงานระหว่างโมเดลภาษาและโมเดลสร้างภาพเข้าด้วยกันแบบไร้รอยต่อ:

```text
[ผู้ใช้ป้อนหัวข้อสั้นๆ (ไทย/อังกฤษ)]
         │
         ▼
┌──────────────────────────────────────────────────┐
│ 1. Jupyter Notebook (Python Orchestrator)        │
└────────┬────────────────────────────────┬────────┘
         │ (1) ส่งหัวข้อไปขยายเป็น Prompt      │ (3) ส่ง JSON Workflow + Seed ใหม่
         ▼                                ▼
┌────────────────────────────┐   ┌─────────────────────────────────┐
│ 2. LM Studio API (:1234)   │   │ 3. ComfyUI API (:8188)          │
│ • โมเดล: Qwen2.5 / Llama 3B│   │ • โมเดล: Stable Diffusion 1.5    │
│ • แปลงไทย -> SD 1.5 Prompt │   │ • เรนเดอร์ภาพ 512x512 (~2-3 วินาที)│
└────────────────────────────┘   └────────────────┬────────────────┘
                                                  │ (4) ดึงภาพผ่าน /history & /view
                                                  ▼
                                 [แสดงผลภาพใน Jupyter Notebook ทันที]
```

---

## 💡 ทริคสำคัญ: การจัดการ VRAM 6GB เมื่อรัน LM Studio + ComfyUI พร้อมกัน

สำหรับเครื่องห้องปฏิบัติการที่ใช้การ์ดจอ **NVIDIA GeForce RTX 3050 (VRAM 6GB)** coupled กับ **Intel Core Ultra 7 265** (หรือเครื่อง macOS Apple Silicon 16GB):
1. **เลือกขนาดโมเดลที่เหมาะสม:** ใช้ LLM ขนาด **3B (`Q4_K_M`)** ใน LM Studio (~2.2 GB) คู่กับ **Stable Diffusion 1.5** ใน ComfyUI (~2.0 GB) ซึ่งใช้ VRAM รวมกันประมาณ ~4.2–4.5 GB ไม่เกินเพดาน 6GB
2. **คุมขนาด Context Length:** ใน LM Studio ตั้งค่า Context Length ไม่เกิน `2048` เพื่อไม่ให้ KV Cache กินพื้นที่การ์ดจอ
3. **แบ่งภาระมาที่ CPU (GPU Offload) & `--lowvram`:**
   - หากพบอาการหน่วยความจำการ์ดจอตึงเกินไปบน Windows สามารถปรับลดสไลเดอร์ **GPU Offload** ใน LM Studio ลงมาที่ระดับ 50%–70% เพื่อให้ซีพียู **Intel Core Ultra 7 265 (20 Cores)** ช่วยประมวลผลบางเลเยอร์
   - ในฝั่ง ComfyUI บน Windows สามารถเติมแฟล็ก `--lowvram` ในไฟล์ `run_nvidia_gpu.bat` เพื่อให้ ComfyUI สลับโมเดลเข้า-ออก VRAM อัตโนมัติ

In [ ]:
import io
import json
import random
import time
import urllib.parse
from pathlib import Path

import requests
from openai import OpenAI
from PIL import Image
from IPython.display import display, Markdown

# ตั้งค่า Endpoint ของทั้งสองเซิร์ฟเวอร์
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
COMFYUI_BASE_URL = "http://127.0.0.1:8188"
WORKFLOW_JSON_PATH = Path("../03-comfyui-workflows/sd15_txt2img_api.json")

llm_client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key="lm-studio")
print("✅ โหลดไลบรารีและตั้งค่าตัวแปรเริ่มต้นเรียบร้อยแล้ว")

## ส่วนที่ 1: ฟังก์ชัน `generate_sd_prompt(topic: str) -> str` (LM Studio)

โมเดล **Stable Diffusion 1.5** เข้าใจคำบรรยายภาษาอังกฤษเชิงทัศนศิลป์ (Visual Keywords / Danbooru-style tags หรือ Descriptive Phrases) ได้ดีที่สุด เราจึงสร้างฟังก์ชัน `generate_sd_prompt()` เพื่อให้ Local LLM ใน LM Studio ทำหน้าที่เป็น **Prompt Engineer** แปลงหัวข้อภาษาไทยสั้นๆ ให้กลายเป็น Positive Prompt ภาษาอังกฤษที่สมบูรณ์

In [ ]:
def generate_sd_prompt(topic: str) -> str:
    """
    รับหัวข้อภาษาไทยหรืออังกฤษสั้นๆ จากผู้ใช้ แล้วส่งให้ LM Studio
    แปลงและขยายเป็น Positive Prompt ภาษาอังกฤษสำหรับ Stable Diffusion 1.5
    """
    models = llm_client.models.list()
    model_id = models.data[0].id if models.data else "local-model"

    system_instruction = (
        "You are an expert Stable Diffusion 1.5 prompt engineer. "
        "Convert the user's Thai or English topic into a single, vivid, comma-separated "
        "English image generation prompt. Include subject details, environment, lighting, "
        "camera angle, and quality tags (masterpiece, best quality, highly detailed, cinematic lighting). "
        "Output ONLY the English prompt text without quotes or explanations."
    )

    response = llm_client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": f"Topic: {topic}"}
        ],
        temperature=0.7,
        max_tokens=180
    )
    return response.choices[0].message.content.strip()

# ทดสอบฟังก์ชันสร้าง Prompt จากหัวข้อภาษาไทย
sample_topic = "ห้องสมุดอนาคตในมหาวิทยาลัยไทยที่มีหุ่นยนต์ผู้ช่วยและแสงไฟนีออนอบอุ่น"
# ยกเลิกคอมเมนต์บรรทัดด้านล่างเมื่อเปิด LM Studio Server แล้วเพื่อทดสอบทันที
# print(generate_sd_prompt(sample_topic))

## ส่วนที่ 2: ฟังก์ชัน `queue_prompt()` และดึงภาพผลลัพธ์จาก ComfyUI API

ในส่วนนี้เราจะโหลดไฟล์โครงสร้างเวิร์กโฟลว์ `../03-comfyui-workflows/sd15_txt2img_api.json` มาแก้ไขค่าผ่านโค้ด Python:
- **Node `"6"` (`CLIPTextEncode`):** แทนที่ `inputs["text"]` ด้วย Prompt ภาษาอังกฤษที่ได้จาก LM Studio
- **Node `"3"` (`KSampler`):** สุ่มค่า `inputs["seed"]` ใหม่ทุกครั้งเพื่อให้ได้ภาพที่ไม่ซ้ำเดิม
- ส่งเข้าคิวประมวลผลที่ `POST http://127.0.0.1:8188/prompt` แล้วรอรับไฟล์ภาพผ่าน `/history/{prompt_id}` และ `/view`

In [ ]:
def queue_prompt(prompt_workflow: dict) -> dict:
    """ส่ง JSON Workflow (API Format) เข้าคิวเรนเดอร์ของ ComfyUI"""
    payload = {"prompt": prompt_workflow}
    response = requests.post(f"{COMFYUI_BASE_URL}/prompt", json=payload, timeout=10)
    response.raise_for_status()
    return response.json()


def wait_and_fetch_image(prompt_id: str, timeout_sec: int = 120) -> Image.Image:
    """รอให้ ComfyUI เรนเดอร์ภาพเสร็จ แล้วดึงไบต์ภาพกลับมาเป็น PIL.Image"""
    start_time = time.time()
    while time.time() - start_time < timeout_sec:
        history_res = requests.get(f"{COMFYUI_BASE_URL}/history/{prompt_id}", timeout=5)
        history_data = history_res.json()

        if prompt_id in history_data:
            outputs = history_data[prompt_id].get("outputs", {})
            # ดึงข้อมูลรูปภาพจาก Node "9" (SaveImage) หรือโหนดใดๆ ที่มี images
            for node_id, node_output in outputs.items():
                if "images" in node_output and len(node_output["images"]) > 0:
                    img_meta = node_output["images"][0]
                    query = urllib.parse.urlencode({
                        "filename": img_meta["filename"],
                        "subfolder": img_meta.get("subfolder", ""),
                        "type": img_meta.get("type", "output")
                    })
                    img_res = requests.get(f"{COMFYUI_BASE_URL}/view?{query}", timeout=10)
                    img_res.raise_for_status()
                    return Image.open(io.BytesIO(img_res.content))
        time.sleep(0.8)

    raise TimeoutError(f"หมดเวลารอผลลัพธ์จาก ComfyUI ({timeout_sec} วินาที)")

## ส่วนที่ 3: รันไปป์ไลน์ครบวงจร (End-to-End Local AI Pipeline)

เพียงใส่หัวข้อภาษาไทยที่คุณต้องการในตัวแปร `user_topic` แล้วรัน Cell ด้านล่าง ระบบจะสั่งการให้ **LM Studio** คิด Prompt และส่งต่อไปให้ **ComfyUI** วาดภาพกลับมาแสดงผลใน Jupyter Notebook โดยอัตโนมัติ!

In [ ]:
def run_local_ai_pipeline(user_topic: str, custom_checkpoint: str = None):
    """ฟังก์ชันหลักสำหรับรันไปป์ไลน์ Jupyter -> LM Studio -> ComfyUI -> Display Image"""
    display(Markdown(f"### 🚀 เริ่มต้นประมวลผลหัวข้อ: **{user_topic}**"))

    # ขั้นตอนที่ 1: ให้ LM Studio แต่ง Positive Prompt ภาษาอังกฤษ
    print("1️⃣ กำลังส่งหัวข้อให้ LM Studio ขยายเป็น Image Prompt ภาษาอังกฤษ...")
    positive_prompt = generate_sd_prompt(user_topic)
    display(Markdown(f"> **✨ Generated SD 1.5 Prompt:** `{positive_prompt}`"))

    # ขั้นตอนที่ 2: โหลดไฟล์ sd15_txt2img_api.json และอัปเดตค่า Node "6" และ Node "3"
    workflow = json.loads(WORKFLOW_JSON_PATH.read_text(encoding="utf-8"))
    workflow["6"]["inputs"]["text"] = positive_prompt
    new_seed = random.randint(1, 999_999_999_999_999)
    workflow["3"]["inputs"]["seed"] = new_seed

    if custom_checkpoint:
        workflow["4"]["inputs"]["ckpt_name"] = custom_checkpoint

    # ขั้นตอนที่ 3: ส่งเข้าคิวเรนเดอร์ใน ComfyUI
    print(f"2️⃣ กำลังส่งเวิร์กโฟลว์เข้าคิว ComfyUI (Seed: {new_seed})...")
    queue_res = queue_prompt(workflow)
    prompt_id = queue_res["prompt_id"]

    # ขั้นตอนที่ 4: รอรับภาพและนำมาแสดงผลบนหน้าจอ Notebook
    print(f"3️⃣ กำลังเรนเดอร์ภาพ (Prompt ID: {prompt_id})...")
    t0 = time.time()
    result_image = wait_and_fetch_image(prompt_id)
    elapsed = time.time() - t0

    print(f"✅ เรนเดอร์ภาพขนาด {result_image.size[0]}x{result_image.size[1]} สำเร็จในเวลา {elapsed:.2f} วินาที!")
    display(result_image)
    return result_image


# =====================================================================
# 🎯 ทดลองเปลี่ยนหัวข้อภาษาไทยตรงนี้แล้วกด Shift + Enter เพื่อรันจริง!
# =====================================================================
user_topic = "แมวส้มใส่แว่นตานักวิทยาศาสตร์กำลังทดลองสร้างหุ่นยนต์ AI ในห้องแล็บมหาวิทยาลัย"

try:
    generated_img = run_local_ai_pipeline(user_topic)
except Exception as err:
    print(f"⚠️ ยังไม่สามารถรันไปป์ไลน์จนจบได้ กรุณาตรวจสอบว่าเปิด LM Studio (:1234) และ ComfyUI (:8188) แล้ว\nรายละเอียด: {err}")